In [1]:
import torch

In [2]:
def drifting_loss(gen: torch.Tensor, pos: torch.Tensor, drift_fn):
    with torch.no_grad():
        V = drift_fn(gen, pos, gen) 
        target = (gen + V).detach()
    return ((gen - target) ** 2).mean()

In [ ]:
def compute_V(X, X_pos, X_neg, *, temp=0.5, mode="gradient",
              ignore_self_neg=True, max_step=None, min_dist=1e-2, eps=1e-8):
    '''Laplace kernel  k(x, y) = exp(-d / temp),  d = ||x - y||.
    Drift  V = (pull toward X_pos) - (push from X_neg):

      base      :  V = E[ k(x,y) (y - x) ] / E[ k(x,y) ]
      gradient  :  V = E[ grad_x k(x,y) ]  / E[ k(x,y) ],
                   with  grad_x k = k(x,y) (y - x) / (temp d)
    '''
    dist_pos = torch.cdist(X, X_pos)
    dist_neg = torch.cdist(X, X_neg)

    k_pos = torch.exp(-dist_pos / temp)
    k_neg = torch.exp(-dist_neg / temp)

    if ignore_self_neg and X.shape[0] == X_neg.shape[0]:   # don't repel a point from itself
        eye = torch.eye(X.shape[0], device=X.device, dtype=torch.bool)
        k_neg = k_neg.masked_fill(eye, 0.0)

    diff_pos = X_pos.unsqueeze(0) - X.unsqueeze(1)         # (y - x), [Nx, Npos, D]
    diff_neg = X_neg.unsqueeze(0) - X.unsqueeze(1)         # (y - x), [Nx, Nneg, D]

                                               
        # V = E[grad_x k] / E[k] -> grad_x k = k (y - x) / (temp d)
    V_pos = (k_pos.unsqueeze(-1) * diff_pos
                / (temp * dist_pos.clamp_min(min_dist)).unsqueeze(-1)).sum(1)
    V_neg = (k_neg.unsqueeze(-1) * diff_neg
                / (temp * dist_neg.clamp_min(min_dist)).unsqueeze(-1)).sum(1)

    V = V_pos - V_neg

    if max_step is not None:
        n = V.norm(dim=-1, keepdim=True)
        V = V * (max_step / n.clamp_min(max_step))
    return V

# Toy 1 — semi-circle: the other half comes for free

Train a generator on the **upper** half of a circle only. The lower half is never in
the training data. It appears anyway, because the architecture cannot represent an
asymmetric output distribution.

**Why.** An O(2)-equivariant map $f:\mathbb{R}^2\to\mathbb{R}^2$ satisfies
$f(Qz) = Qf(z)$ for every orthogonal $Q$. Take $Q$ to be the reflection that fixes
$z$: then $f(z) = Qf(z)$, so $f(z)$ must lie on the ray through $z$. The only such
maps are

$$f(z) \;=\; \rho(\lVert z\rVert)\,\frac{z}{\lVert z\rVert}$$

with $\rho$ a scalar function of the radius alone. The noise is an isotropic
Gaussian, which is O(2)-invariant, so the generated distribution is O(2)-invariant
too — exactly rotationally symmetric, by construction.

The drift field is built from a Laplace kernel on pairwise distances, which is also
O(2)-invariant. Minimizing a kernel discrepancy over an invariant model class lands
on the group-symmetrization of the target, and the symmetrization of an upper arc is
the full ring. So the training signal is consistent with what the architecture can
express. Concretely: every angle shares the one radial profile $\rho$, so the
upper-arc data teaches $\rho(r)\to 1$ and every angle inherits it.

**Be precise about the claim.** The model *learns* the radial profile. The angular
completion is *architectural*, not inferred from data. A plain MLP trained on the
same data is included as a control, and covers only the upper arc.

In [ ]:
import math

import matplotlib.pyplot as plt
from torch import nn

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
torch.manual_seed(7)

RADIUS = 1.0        # circle radius
WIDTH = 0.05        # radial jitter of the data
NOISE_SCALE = 1.0   # standard deviation of the isotropic Gaussian noise

print("device:", device)

In [ ]:
def sample_arc(count, angle_low, angle_high, radius=RADIUS, width=WIDTH):
    """Points on an arc of `radius`, angles uniform on [angle_low, angle_high]."""
    angle = angle_low + (angle_high - angle_low) * torch.rand(count, device=device)
    r = radius + width * torch.randn(count, device=device)
    return r[:, None] * torch.stack((angle.cos(), angle.sin()), dim=1)


def sample_train(count):
    """The ONLY data the models ever see: the upper half of the circle."""
    return sample_arc(count, 0.0, math.pi)


def sample_full(count):
    """Full circle. Reference for plots and metrics; never fed to training."""
    return sample_arc(count, 0.0, 2 * math.pi)


def sample_noise(count):
    return NOISE_SCALE * torch.randn(count, 2, device=device)

In [ ]:
class O2Generator(nn.Module):
    """f(z) = rho(||z||) * z / ||z||  -- the general O(2)-equivariant map R^2 -> R^2.

    The scalar branch predicts the OUTPUT RADIUS directly rather than a scale
    factor. Much better conditioned: the target is rho ~ 1 instead of s ~ 1/r,
    which blows up near the origin.
    """

    def __init__(self, hidden=128, basis=16, max_radius=4.0):
        super().__init__()
        self.register_buffer("centers", torch.linspace(0.0, max_radius, basis))
        self.gamma = 1.0 / (max_radius / max(basis - 1, 1)) ** 2
        self.net = nn.Sequential(
            nn.Linear(basis + 1, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 1),
        )

    def radius_profile(self, r):
        """rho(r), the entire learned content of this model."""
        radial = torch.exp(-self.gamma * (r - self.centers).square())
        return nn.functional.softplus(self.net(torch.cat((r, radial), dim=-1)))

    def forward(self, z):
        r = z.norm(dim=-1, keepdim=True).clamp_min(1e-6)
        return self.radius_profile(r) * z / r


class PlainGenerator(nn.Module):
    """Unconstrained control. Same data, same loss, no symmetry."""

    def __init__(self, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 2),
        )

    def forward(self, z):
        return self.net(z)

In [ ]:
def random_o2(count, reflect):
    """Random elements of O(2): rotations, or rotations composed with a mirror."""
    angle = 2 * math.pi * torch.rand(count, device=device)
    cos, sin = angle.cos(), angle.sin()
    Q = torch.stack((torch.stack((cos, -sin), -1), torch.stack((sin, cos), -1)), -2)
    if reflect:
        Q = Q @ torch.diag(torch.tensor([1.0, -1.0], device=device))
    return Q


def equivariance_error(model, count=512):
    """max |f(Qz) - Q f(z)| over random rotations and reflections."""
    z = sample_noise(count)
    worst = 0.0
    for reflect in (False, True):
        Q = random_o2(count, reflect)
        with torch.no_grad():
            left = model((Q @ z[..., None]).squeeze(-1))
            right = (Q @ model(z)[..., None]).squeeze(-1)
        worst = max(worst, float((left - right).abs().max()))
    return worst


equivariant_error = equivariance_error(O2Generator().to(device))
control_error = equivariance_error(PlainGenerator().to(device))
print(f"O2Generator    max |f(Qz) - Q f(z)| = {equivariant_error:.3e}")
print(f"PlainGenerator max |f(Qz) - Q f(z)| = {control_error:.3e}")
assert equivariant_error < 1e-5, "O2Generator is not equivariant"
assert control_error > 1e-2, "control is unexpectedly symmetric"

In [ ]:
HELD_OUT = "#c1440e"   # the withheld lower arc
SEEN = "#1f6feb"       # data the model is trained on
GENERATED = "#0f7b6c"


def circle_xy(angle_low, angle_high, radius=RADIUS, steps=200):
    angle = torch.linspace(angle_low, angle_high, steps)
    return radius * angle.cos(), radius * angle.sin()


def style_axes(ax, limit=1.6):
    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit)
    ax.set_aspect("equal")
    ax.axhline(0.0, color="0.8", lw=0.8, zorder=0)
    ax.set_xticks([]), ax.set_yticks([])
    for side in ax.spines.values():
        side.set_color("0.85")


def draw_reference(ax):
    """Solid where data exists, dashed red where it was withheld."""
    ax.plot(*circle_xy(0, math.pi), color=SEEN, lw=1.2, alpha=0.5, zorder=1)
    ax.plot(*circle_xy(math.pi, 2 * math.pi), color=HELD_OUT, lw=1.4,
            ls="--", alpha=0.9, zorder=1)


def scatter(ax, points, color, title, size=3, alpha=0.25):
    p = points.detach().cpu()
    ax.scatter(p[:, 0], p[:, 1], s=size, c=color, alpha=alpha, linewidths=0)
    ax.set_title(title, fontsize=10)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(9, 4.6))

scatter(axes[0], sample_noise(6000), "0.45",
        f"Input: isotropic Gaussian noise (sigma = {NOISE_SCALE})", alpha=0.18)
style_axes(axes[0], limit=3.4)

scatter(axes[1], sample_train(6000), SEEN, "Training data: upper arc only")
draw_reference(axes[1])
axes[1].text(0.0, -1.28, "withheld", color=HELD_OUT, ha="center", fontsize=9)
style_axes(axes[1])

figure.tight_layout()
figure.subplots_adjust(top=0.85)
figure.suptitle("Before training", fontsize=12)
plt.show()

In [ ]:
TEMP = 0.4


def drift_fn(gen, pos, neg):
    """compute_V, mean-normalized over the reference batch.

    compute_V sums kernel contributions rather than averaging them, so its
    magnitude scales with batch size. Dividing by the batch restores the E[.]
    the docstring describes and makes training stable. Clamping with `max_step`
    instead of normalizing also trains, but diverged on 2 of 5 seeds here.
    """
    return compute_V(gen, pos, neg, temp=TEMP) / pos.shape[0]


def train(model, steps=4000, batch=512, lr=2e-3, report_every=1000):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for step in range(steps + 1):
        generated = model(sample_noise(batch))
        loss = drifting_loss(generated, sample_train(batch), drift_fn)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach()))
        if step % report_every == 0:
            print(f"  step {step:5d}   loss {history[-1]:.3e}")
    return history


torch.manual_seed(7)
equivariant = O2Generator().to(device)
print("O2Generator")
equivariant_history = train(equivariant)

torch.manual_seed(7)
control = PlainGenerator().to(device)
print("PlainGenerator (control)")
control_history = train(control)

with torch.no_grad():
    equivariant_samples = equivariant(sample_noise(20_000))
    control_samples = control(sample_noise(20_000))

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(13, 4.8))

scatter(axes[0], sample_train(6000), SEEN, "Training data (upper arc only)")
draw_reference(axes[0])
style_axes(axes[0])

scatter(axes[1], equivariant_samples[:6000], GENERATED,
        "O(2)-equivariant generator")
draw_reference(axes[1])
style_axes(axes[1])

scatter(axes[2], control_samples[:6000], GENERATED, "Control: plain MLP")
draw_reference(axes[2])
style_axes(axes[2])

for ax in axes[1:]:
    ax.axhspan(-1.6, 0.0, color=HELD_OUT, alpha=0.05, zorder=0)
    ax.text(0.0, -1.45, "never in training data", color=HELD_OUT,
            ha="center", fontsize=8)

figure.tight_layout()
figure.subplots_adjust(top=0.85)
figure.suptitle("After training on the upper arc only", fontsize=12)
plt.show()

In [ ]:
BINS = 36


def angular_histogram(samples):
    angle = torch.atan2(samples[:, 1], samples[:, 0]) % (2 * math.pi)
    return torch.histc(angle, bins=BINS, min=0.0, max=2 * math.pi) / len(samples)


def metrics(samples):
    counts = angular_histogram(samples)
    return {
        "held-out mass (y < 0)": float((samples[:, 1] < 0).float().mean()),
        "radius error |r - 1|": float((samples.norm(dim=-1) - RADIUS).abs().mean()),
        "empty angular bins": float((counts < 1e-4).float().mean()),
        "max deviation from uniform angle": float((counts - 1 / BINS).abs().max()),
    }


rows = {
    "O(2)-equivariant": metrics(equivariant_samples),
    "control (plain MLP)": metrics(control_samples),
    "reference (full circle)": metrics(sample_full(20_000)),
}

print(f"{'':34s}" + "".join(f"{key:>24s}" for key in rows))
for name in next(iter(rows.values())):
    print(f"{name:34s}" + "".join(f"{rows[key][name]:>24.4f}" for key in rows))
print()
print("The held-out mass is the headline: the equivariant model puts half its")
print("samples on an arc it was never shown. The control puts essentially none.")

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(13, 3.8))

# Angular coverage: where the mass actually landed.
width = 2 * math.pi / BINS
edges = torch.linspace(0, 2 * math.pi, BINS + 1)[:-1] + width / 2
equivariant_counts = angular_histogram(equivariant_samples).cpu()
control_counts = angular_histogram(control_samples).cpu()
axes[0].bar(edges, equivariant_counts, width=width, color=GENERATED,
            alpha=0.8, label="equivariant")
axes[0].bar(edges, control_counts, width=width, color=HELD_OUT,
            alpha=0.5, label="control")
axes[0].axhline(1 / BINS, color="0.35", ls=":", lw=1, label="uniform")
axes[0].axvspan(math.pi, 2 * math.pi, color=HELD_OUT, alpha=0.06)
ceiling = float(max(equivariant_counts.max(), control_counts.max()))
axes[0].set_ylim(0, ceiling * 1.55)
axes[0].text(1.5 * math.pi, ceiling * 1.12, "held out", color=HELD_OUT,
             ha="center", fontsize=8)
axes[0].set_xlabel("angle (radians)"), axes[0].set_ylabel("fraction of samples")
axes[0].set_title("Angular coverage", fontsize=10)
axes[0].legend(fontsize=8, frameon=False, loc="upper left", ncol=3,
               columnspacing=1.0, handlelength=1.2)

# The one scalar function the equivariant model learned.
grid = torch.linspace(0.01, 3.5, 400, device=device)[:, None]
with torch.no_grad():
    rho = equivariant.radius_profile(grid).squeeze(-1).cpu()
twin = axes[1].twinx()
twin.hist(sample_noise(20_000).norm(dim=-1).cpu(), bins=80, color="0.8", alpha=0.5)
twin.set_yticks([])
axes[1].plot(grid.cpu(), rho, color=GENERATED, lw=1.8, zorder=3)
axes[1].axhline(RADIUS, color="0.35", ls=":", lw=1, zorder=2)
axes[1].set_zorder(twin.get_zorder() + 1)
axes[1].patch.set_visible(False)
axes[1].set_xlabel("noise radius  r"), axes[1].set_ylabel("output radius  rho(r)")
axes[1].set_title("Learned radial profile (grey: noise radii)", fontsize=10)

axes[2].plot(equivariant_history, color=GENERATED, lw=0.8, label="equivariant")
axes[2].plot(control_history, color=HELD_OUT, lw=0.8, alpha=0.7, label="control")
axes[2].set_yscale("log")
axes[2].set_xlabel("step"), axes[2].set_ylabel("drifting loss")
axes[2].set_title("Training loss", fontsize=10)
axes[2].legend(fontsize=8, frameon=False)

for ax in (*axes, twin):
    for side in ax.spines.values():
        side.set_color("0.85")
figure.tight_layout()
plt.show()

## What this does and does not show

The equivariant generator recovers the full circle from upper-arc data, with half its
mass on an arc it never saw and a radius error matching the data's own jitter. The
control, same loss and same data, leaves the lower half empty.

But note *where* the symmetry lives. The middle panel above is the model's entire
learned content: one scalar function of the radius. The angular uniformity is not
learned at all, it is forced by the architecture. That is the point of the toy, and
also its limit.

**Why the mass splits 50/50.** The Laplace kernel in `compute_V` depends only on
pairwise distances, so it is O(2)-invariant. Minimizing a kernel discrepancy over an
invariant model class converges to the group-symmetrization of the target, and the
O(2)-symmetrization of the upper arc is the full ring at half the density. The pull
toward the upper arc is exactly balanced by repulsion within it.

**The limit, and the next step.** Full O(2) can *only* produce rotationally symmetric
output, so this same architecture cannot do a flower, a star, or a letter. Richer
shapes need a *smaller* group — a finite subgroup `C_k` or `D_k` — built by
canonicalization: rotate the input into a fundamental wedge, apply an unconstrained
MLP there, rotate the output back. That keeps exact equivariance while leaving the
model free to shape the wedge. Same experiment, one petal in, the whole flower out.